# 04 Continual Learning

Purpose: Rebuild an isolated ChromaDB at chroma_db_continual from scratch, then evaluate all five RAC scoring variants as the DB grows from 10% to 100%.

Inputs: dataset/CVPR_2024_dataset_Train/, dataset/CVPR_2024_dataset_Test/, dataset_text/train.csv, dataset_text/test.csv.

Outputs: results/phase2/continual_learning_results.json, figures/phase2/continual_learning_curve.png.

In [1]:
import math
import shutil
import sys
from pathlib import Path

import numpy as np
from PIL import Image
from tqdm.auto import tqdm

sys.path.insert(0, str(Path("../..").resolve()))

from src.phase2.config import get_phase2_config
from src.phase2.data_utils import build_records_from_csv
from src.phase2.db_client import (
    get_image_collection,
    get_persistent_client,
    get_text_collection,
)
from src.phase2.evaluation import (
    evaluate_variant,
    save_continual_summary_csv,
    save_results,
)
from src.phase2.gpu_utils import get_device, print_device_info, print_gpu_memory
from src.phase2.scoring import global_dnds, idw, kde_dnds, local_dnds, majority_vote
from src.phase2.visualization import plot_continual_learning_curve

CONFIG = get_phase2_config({"continual_db_path": "./chroma_db_continual"})

# Per-notebook GPU controls
PREFER_GPU = True
CLEANUP_INTERVAL = 0
MEMORY_LOG_INTERVAL = 0

DEVICE = get_device(prefer_gpu=PREFER_GPU)
print_device_info(DEVICE)
if MEMORY_LOG_INTERVAL > 0:
    print_gpu_memory(prefix="Startup GPU memory", device=DEVICE)

REPO_ROOT = Path("../..").resolve()
TRAIN_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Train"
TEST_DIR = REPO_ROOT / "dataset" / "CVPR_2024_dataset_Test"
TRAIN_CSV = REPO_ROOT / "dataset_text" / "train.csv"
TEST_CSV = REPO_ROOT / "dataset_text" / "test.csv"
RESULTS_PATH = REPO_ROOT / "results" / "phase2" / "continual_learning_results.json"
CONTINUAL_CSV_PATH = REPO_ROOT / "results" / "phase2" / "continual_learning_curve.csv"
FIG_PATH = REPO_ROOT / "figures" / "phase2" / "continual_learning_curve.png"

for required in [TRAIN_DIR, TEST_DIR, TRAIN_CSV, TEST_CSV]:
    if not required.exists():
        raise FileNotFoundError(
            f"Missing required input for continual learning notebook: {required}"
        )

train_records, train_missing_examples, train_total_rows = build_records_from_csv(
    csv_path=TRAIN_CSV,
    split_dir=TRAIN_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)
test_samples, test_missing_examples, test_total_rows = build_records_from_csv(
    csv_path=TEST_CSV,
    split_dir=TEST_DIR,
    text_column="text",
    label_column="label",
    text_key="text",
)

if train_missing_examples:
    print("Skipped train rows with missing image files (up to 10 shown):")
    for item in train_missing_examples:
        print(f"  - {item}")

if test_missing_examples:
    print("Skipped test rows with missing image files (up to 10 shown):")
    for item in test_missing_examples:
        print(f"  - {item}")

if not train_records:
    raise RuntimeError("No train records found for continual-learning simulation.")
if not test_samples:
    raise RuntimeError("No test records found for continual-learning evaluation.")

print(f"Train records used: {len(train_records)} / {train_total_rows}")
print(f"Test samples used: {len(test_samples)} / {test_total_rows}")

variant_fns = {
    "majority_vote": majority_vote,
    "idw": idw,
    "global_dnds": global_dnds,
    "local_dnds": local_dnds,
    "kde_dnds": kde_dnds,
}

d:\University of Calgary\Winter 2026\ENSF617-Introduction-To-Machine-Learning\trash-classification-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU (4.00 GB VRAM, allocated 0.00 GB, reserved 0.00 GB)
Train records used: 11685 / 11685
Test samples used: 3452 / 3452


In [ ]:
continual_db_dir = REPO_ROOT / "chroma_db_continual"
if continual_db_dir.exists():
    shutil.rmtree(continual_db_dir)

try:
    client = get_persistent_client(str(continual_db_dir))
    image_collection = get_image_collection(client)
    text_collection = get_text_collection(client)
except Exception as exc:
    raise RuntimeError(f"Failed to initialize isolated continual DB: {exc}") from exc


def add_records_to_db(records: list[dict], start_index: int) -> None:
    batch_size = CONFIG["batch_size"]
    for start in range(0, len(records), batch_size):
        batch = records[start : start + batch_size]
        ids_img, ids_txt, images, docs, metadatas = [], [], [], [], []

        for offset, record in enumerate(batch):
            global_idx = start_index + start + offset
            image_np = np.array(
                Image.open(record["image_path"]).convert("RGB"), dtype=np.uint8
            )
            ids_img.append(f"img_{global_idx}")
            ids_txt.append(f"txt_{global_idx}")
            images.append(image_np)
            docs.append(record["text"])
            metadatas.append({"label": record["label"], "filename": record["text"]})

        try:
            image_collection.add(ids=ids_img, images=images, metadatas=metadatas)
            text_collection.add(ids=ids_txt, documents=docs, metadatas=metadatas)
        except Exception as exc:
            raise RuntimeError(
                f"Failed adding records to continual DB at start={start_index + start}: {exc}"
            ) from exc


results = {
    "db_size_percent": [],
    "variants": {name: {"macro_f1": [], "steps": {}} for name in variant_fns},
}

added_count = 0
total_train = len(train_records)
for pct in tqdm(range(10, 101, 10), desc="Continual growth steps"):
    target_count = math.ceil(total_train * (pct / 100.0))
    target_count = min(target_count, total_train)

    if target_count > added_count:
        new_slice = train_records[added_count:target_count]
        add_records_to_db(new_slice, start_index=added_count)
        added_count = target_count

    image_count = image_collection.count()
    text_count = text_collection.count()
    if image_count != added_count or text_count != added_count:
        raise AssertionError(
            f"Collection counts mismatch at {pct}%: image={image_count}, text={text_count}, expected={added_count}"
        )

    results["db_size_percent"].append(pct)
    for variant_name, score_fn in variant_fns.items():
        metrics = evaluate_variant(
            score_fn,
            test_samples,
            image_collection,
            text_collection,
            CONFIG,
            cleanup_interval=CLEANUP_INTERVAL,
            memory_log_interval=MEMORY_LOG_INTERVAL,
        )
        results["variants"][variant_name]["macro_f1"].append(metrics["macro_f1"])
        results["variants"][variant_name]["steps"][str(pct)] = {
            "db_count": added_count,
            "accuracy": metrics["accuracy"],
            "macro_f1": metrics["macro_f1"],
            "weighted_f1": metrics["weighted_f1"],
            "inference_time_ms": metrics.get("inference_time_ms", 0.0),
        }

if image_collection.count() != total_train or text_collection.count() != total_train:
    raise AssertionError(
        f"Final 100% step must contain the complete training set. "
        f"image={image_collection.count()}, text={text_collection.count()}, expected={total_train}"
    )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 737.15it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Continual growth steps:  10%|█         | 1/10 [43:50<6:34:37, 2630.87s/it]

In [ ]:
save_results(results, str(RESULTS_PATH))
save_continual_summary_csv(results, str(CONTINUAL_CSV_PATH))
plot_continual_learning_curve(results, str(FIG_PATH))

print("Continual learning experiment complete.")
print(f"Saved JSON results: {RESULTS_PATH}")
print(f"Saved CSV curve: {CONTINUAL_CSV_PATH}")
print(f"Saved figure: {FIG_PATH}")
results